# SkillGraph Week 3 — Negation Filtering, Skill Gap Analysis, Priority Modeling & Backend Verification

**Research Project:** *SkillGraph: An Explainable Knowledge Graph-Based Framework for Career-Oriented Skill Gap Analysis and Learning Path Recommendation*

**Notebook Objectives:**
1. Evaluate Module 1 dependency-based negation and clause modality classification (`SpacySkillExtractor`, `DictionarySkillExtractor`, conservative parser-unavailable fallback, 22-doc technical benchmark, and 2,000-doc corpus modality distribution).
2. Evaluate Module 2–5 skill gap analysis, worked numerical priority scoring (`P(s)`), cross-occupation opportunity (`O(s)`), weight sensitivity across `BASELINE_WEIGHT_PRESETS`, and non-fabricated learning-path ordering.
3. Verify Module 6–8 relational SQL persistence (PostgreSQL 17 host audit + 17-table SQLite execution mirror) and REST API ASGI endpoints (`/health`, `/occupations`, `/skills`, `/skill-gap/analyze`, `/recommendations/{student_id}`, `/openapi.json`).
4. Persist measured Week 3 benchmark extensions cleanly into `outputs/reports/skill_extraction_benchmark_report.json` while preserving all Week 1 & Week 2 baseline keys.

> **Research Integrity Notice:** All job postings (`2,000` global synthetic corpus + `22` technical benchmark postings) and demo student profiles used below are **synthetic development data**. The 50-record human sample (`data/annotations/human_annotation_sample_50.json`, `530` candidate spans) remains `PENDING` independent human confirmation and is **excluded** from benchmark evaluation. Curriculum prerequisite edges (`PREREQUISITE_OF`) and university course mappings remain `UNAVAILABLE_PENDING_CURRICULUM_DATA` and are never fabricated from market `CO_OCCURS_WITH` counts.

In [1]:
import json
import sys
import time
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.skill_extraction import (
    DictionarySkillExtractor,
    SpacySkillExtractor,
    SkillExtractionEvaluator,
)
from scripts.build_graph import (
    BASELINE_WEIGHT_PRESETS,
    MultiOccupationAnalyzer,
    POSTGRESQL_DDL_STATEMENTS,
    PriorityWeights,
    RelationalDatabaseManager,
    SkillGapAnalysisEngine,
    SkillGraphBuilder,
    inspect_postgresql_environment,
)
from scripts import generate_synthetic_benchmark as tech_benchmark
from backend.app.main import SkillGraphTestClient, app

TAXONOMY_CSV = PROJECT_ROOT / "data" / "processed" / "global_skill_taxonomy.csv"
TAXONOMY_JSON = PROJECT_ROOT / "outputs" / "exports" / "skill_taxonomy.json"
GLOBAL_GT_PATH = PROJECT_ROOT / "data" / "annotations" / "synthetic_job_postings_ground_truth.json"
BENCHMARK_REPORT_PATH = PROJECT_ROOT / "outputs" / "reports" / "skill_extraction_benchmark_report.json"

dict_json = DictionarySkillExtractor(TAXONOMY_JSON)
spacy_json = SpacySkillExtractor(TAXONOMY_JSON, use_fast_tokenizer=True)
dict_global = DictionarySkillExtractor(TAXONOMY_CSV)
spacy_global = SpacySkillExtractor(TAXONOMY_CSV, use_fast_tokenizer=True)

builder = SkillGraphBuilder(include_broad_country_mappings=False, min_co_occurrence=5)
graph_summary = builder.build()
engine = SkillGapAnalysisEngine(builder)
multi_occ = MultiOccupationAnalyzer(builder)

print("Project root:", PROJECT_ROOT)
print("Graph nodes:", graph_summary["total_active_nodes"], "| Graph edges:", graph_summary["total_active_edges"])

Project root: D:\Downloads\SkillGraph
Graph nodes: 2336 | Graph edges: 31503


## 1. Module 1 — Dependency-Based Negation & Clause Modality Evaluation

Tests both clause-scoped regex classification (`DictionarySkillExtractor`) and spaCy dependency-parse governor traversal (`SpacySkillExtractor`) across the 8 prompt test cases, `SYN_JOB_003`, the non-taxonomy prohibited clause, parser-unavailable fallback, the 22-document technical benchmark, and the 2,000-document synthetic corpus.

In [2]:
negation_cases = [
    {"text": "Must have Python experience.", "expected_skills": {"Python": ("required", False)}},
    {"text": "Python is preferred.", "expected_skills": {"Python": ("preferred", False)}},
    {"text": "Python is not required.", "expected_skills": {"Python": ("negated_requirement", True)}},
    {"text": "Do not use Python.", "expected_skills": {"Python": ("excluded_or_prohibited", True)}},
    {"text": "Experience with Python is unnecessary.", "expected_skills": {"Python": ("negated_requirement", True)}},
    {"text": "Knowledge of Python and Java is required.", "expected_skills": {"Python": ("required", False), "Java": ("required", False)}},
    {"text": "Python is not required, but SQL is essential.", "expected_skills": {"Python": ("negated_requirement", True), "SQL": ("required", False)}},
    {"text": "Note that we do not develop in Python or Java for this position.", "expected_skills": {"Python": ("excluded_or_prohibited", True), "Java": ("excluded_or_prohibited", True)}},
]

spacy_neg_eval = SkillExtractionEvaluator.evaluate_negation_cases(negation_cases, spacy_json)
dict_neg_eval = SkillExtractionEvaluator.evaluate_negation_cases(negation_cases, dict_json)

# Case 8: non-taxonomy prohibited clause
prohib_analysis = spacy_json.analyze_sentences_and_negation("Candidates must avoid using deprecated technologies.")
prohib_clause = prohib_analysis["non_taxonomy_prohibited_clauses"][0]

# Fallback check when dependency parser is disabled
spacy_fallback = SpacySkillExtractor(TAXONOMY_JSON, use_fast_tokenizer=True)
spacy_fallback.has_dependency_parser = False
fallback_analysis = spacy_fallback.analyze_sentences_and_negation("Python is not required, but SQL is essential.")

print(f"spaCy 8-case negation assertions: {spacy_neg_eval['passed_assertions']}/{spacy_neg_eval['total_assertions']} (acc={spacy_neg_eval['accuracy']:.4f})")
print(f"Dict  8-case negation assertions: {dict_neg_eval['passed_assertions']}/{dict_neg_eval['total_assertions']} (acc={dict_neg_eval['accuracy']:.4f})")
print("Non-taxonomy prohibited clause:", prohib_clause["modality"], "| target:", prohib_clause["target_phrase"], "| skill_id:", prohib_clause["skill_id"])
print("Parser-unavailable fallback method:", fallback_analysis["analysis_method"])

display(pd.DataFrame(spacy_neg_eval["details"])[["skill", "expected_modality", "predicted_modality", "expected_is_negated", "predicted_is_negated", "passed"]])

spaCy 8-case negation assertions: 11/11 (acc=1.0000)
Dict  8-case negation assertions: 11/11 (acc=1.0000)
Non-taxonomy prohibited clause: excluded_or_prohibited | target: deprecated technologies | skill_id: None
Parser-unavailable fallback method: conservative_regex_fallback_parser_unavailable


,skill,expected_modality,predicted_modality,expected_is_negated,predicted_is_negated,passed
0,Python,required,required,False,False,True
1,Python,preferred,preferred,False,False,True
2,Python,negated_requirement,negated_requirement,True,True,True
3,Python,excluded_or_prohibited,excluded_or_prohibited,True,True,True
4,Python,negated_requirement,negated_requirement,True,True,True
5,Python,required,required,False,False,True
6,Java,required,required,False,False,True
7,Python,negated_requirement,negated_requirement,True,True,True
8,SQL,required,required,False,False,True
9,Python,excluded_or_prohibited,excluded_or_prohibited,True,True,True


In [3]:
# 22-doc technical benchmark before vs after Week 3 negation filtering
tech_22_results = {}
for label, ext in [("DictionarySkillExtractor", dict_json), ("SpacySkillExtractor", spacy_json)]:
    tp = fp = fn = 0
    tp_incl = fp_incl = fn_incl = 0
    for job in tech_benchmark.SYNTHETIC_JOBS:
        gt = set(job["ground_truth_skills"])
        preds = {sp["canonical_skill"] for sp in ext.extract(job["description"], include_negated=False)}
        preds_incl = {sp["canonical_skill"] for sp in ext.extract(job["description"], include_negated=True)}
        tp += len(preds & gt)
        fp += len(preds - gt)
        fn += len(gt - preds)
        tp_incl += len(preds_incl & gt)
        fp_incl += len(preds_incl - gt)
        fn_incl += len(gt - preds_incl)
    p = tp / (tp + fp)
    r = tp / (tp + fn)
    f1 = 2 * p * r / (p + r)
    tech_22_results[label] = {
        "exclude_negated_true": {"tp": tp, "fp": fp, "fn": fn, "precision": round(p, 6), "recall": round(r, 6), "f1": round(f1, 6)},
        "include_negated_true": {"tp": tp_incl, "fp": fp_incl, "fn": fn_incl},
    }
    print(f"{label} (22 tech docs): include_negated=False -> TP={tp}, FP={fp}, FN={fn}, P={p:.6f}, R={r:.6f}, F1={f1:.6f} | include_negated=True -> TP={tp_incl}, FP={fp_incl}, FN={fn_incl}")

# 2,000-doc corpus evaluation & modality distribution
with open(GLOBAL_GT_PATH, "r", encoding="utf-8-sig") as f:
    gt_payload = json.load(f)
global_items = gt_payload["data"] if isinstance(gt_payload, dict) and "data" in gt_payload else gt_payload
gt_by_job = {item["job_id"]: item["annotations"] for item in global_items}

t0 = time.perf_counter()
spacy_preds_pos = {item["job_id"]: spacy_global.extract(item["text"], include_negated=False) for item in global_items}
spacy_elapsed = round(time.perf_counter() - t0, 4)
spacy_preds_all = {item["job_id"]: spacy_global.extract(item["text"], include_negated=True) for item in global_items}

corpus_exact_eval = SkillExtractionEvaluator.evaluate_exact_spans(spacy_preds_pos, gt_by_job)
corpus_modality_counts = SkillExtractionEvaluator.summarize_modality_distribution(spacy_preds_all)

print(f"\n2,000-doc strict evaluation (SpacySkillExtractor, {spacy_elapsed}s): TP={corpus_exact_eval['total_tp']}, FP={corpus_exact_eval['total_fp']}, FN={corpus_exact_eval['total_fn']}, Precision={corpus_exact_eval['micro_precision']:.6f}, Recall={corpus_exact_eval['micro_recall']:.6f}, F1={corpus_exact_eval['micro_f1']:.6f}")
print("2,000-doc corpus modality distribution:", corpus_modality_counts)

DictionarySkillExtractor (22 tech docs): include_negated=False -> TP=156, FP=0, FN=0, P=1.000000, R=1.000000, F1=1.000000 | include_negated=True -> TP=156, FP=2, FN=0
SpacySkillExtractor (22 tech docs): include_negated=False -> TP=156, FP=0, FN=0, P=1.000000, R=1.000000, F1=1.000000 | include_negated=True -> TP=156, FP=2, FN=0



2,000-doc strict evaluation (SpacySkillExtractor, 1.1647s): TP=21563, FP=1611, FN=151, Precision=0.930482, Recall=0.993046, F1=0.960747
2,000-doc corpus modality distribution: {'contextually_irrelevant': 728, 'mentioned': 858, 'preferred': 5769, 'required': 15819}


## 2. Modules 2, 3 & 5 — Skill Gap Analysis, Worked Numerical Calculation & Weight Sensitivity

Demonstrates target-occupation coverage metrics, explicit handling of empty/partial/complete/edge student profiles, a step-by-step mathematical verification of `P(s) = clamp((alpha*G(s) + beta*O(s) + gamma_eff*D(s) - delta_eff*C(s)) / (alpha + beta + gamma_eff), 0, 1)`, and rank responsiveness across the four transparent `BASELINE_WEIGHT_PRESETS`.

In [4]:
# Define 4 synthetic profiles testing diversity & edge cases
da_skills = builder.to_relational_tables()["edge_OCCUPATION_DEMANDS_SKILL"]
da_skill_ids = da_skills[da_skills["occupation_id"] == "SG-OC-002"]["skill_id"].tolist()

profiles = {
    "empty_ds": {
        "student_id": "SYNTHETIC_EMPTY_00",
        "is_synthetic": True,
        "target_occupation": "Data Scientist",
        "skills": [],
    },
    "partial_da": {
        "student_id": "SYNTHETIC_STUDENT_001",
        "is_synthetic": True,
        "target_occupation": "Data Analyst",
        "skills": [
            {"skill": "Python", "proficiency_level": "intermediate"},
            {"skill": "SQL", "proficiency_level": "beginner"},
            {"skill": "Communication", "proficiency_level": "advanced"},
        ],
    },
    "edge_bi": {
        "student_id": "SYNTHETIC_EDGE_02",
        "is_synthetic": True,
        "target_occupation": "Business Intelligence Analyst",
        "skills": [
            "Python",
            {"skill": "SQL", "proficiency_score": 0.35},
            {"skill": "postgres", "proficiency_level": "intermediate"},
            "NonExistentSkillXYZ",
        ],
    },
    "complete_da": {
        "student_id": "SYNTHETIC_COMPLETE_DA",
        "is_synthetic": True,
        "target_occupation": "Data Analyst",
        "skills": [{"skill_id": sid, "proficiency_level": "advanced"} for sid in da_skill_ids],
    },
}

profile_analyses = {k: engine.analyze(v) for k, v in profiles.items()}
summary_rows = []
for name, res in profile_analyses.items():
    cm = res["coverage_metrics"]
    summary_rows.append({
        "profile": name,
        "target_occupation": res["target_occupation"]["occupation"],
        "required_coverage": cm["required_skill_coverage"],
        "overall_weighted_coverage": cm["overall_weighted_coverage"],
        "skill_gap_score": cm["skill_gap_score"],
        "possessed": cm["possessed_count"],
        "partial": cm["partially_demonstrated_count"],
        "missing_req": cm["missing_required_count"],
        "missing_pref": cm["missing_preferred_count"],
        "unrecognized": len(res["unrecognized_skills"]),
    })

display(pd.DataFrame(summary_rows))

,profile,target_occupation,required_coverage,overall_weighted_coverage,skill_gap_score,possessed,partial,missing_req,missing_pref,unrecognized
0,empty_ds,Data Scientist,0.0000,0.0000,1.0000,0,0,22,7,0
1,partial_da,Data Analyst,0.1190,0.1806,0.8194,2,1,18,5,0
2,edge_bi,Business Intelligence Analyst,0.0208,0.0836,0.9164,1,1,23,4,1
3,complete_da,Data Analyst,1.0000,1.0000,0.0000,26,0,0,0,0


In [5]:
# Worked step-by-step mathematical verification on SYNTHETIC_STUDENT_001 -> Data Analyst
partial_res = profile_analyses["partial_da"]
worked_rows = []
for r in partial_res["priority_recommendations"][:8]:
    comp = r["scoring"]["components"]
    w = r["scoring"]["weights_applied"]
    opp = r["opportunity"]
    g_check = round(r["gap_factor"] * (1.0 * r["required_share"] + 0.5 * r["preferred_share"]), 6)
    o_check = round(opp["raw_external_opportunity"] / opp["max_external_normalizer"], 4)
    p_check = round(w["alpha"] * comp["G_gap_importance"] + w["beta"] * comp["O_external_opportunity"], 4)
    assert abs(g_check - comp["G_gap_importance"]) < 1e-4
    assert abs(o_check - comp["O_external_opportunity"]) < 5e-4
    assert abs(p_check - r["priority_score"]) < 5e-4
    worked_rows.append({
        "rank": r["rank"],
        "skill": r["canonical_skill"],
        "req_type": r["requirement_type"],
        "gap_factor": r["gap_factor"],
        "req_share": r["required_share"],
        "pref_share": r["preferred_share"],
        "G(s)": comp["G_gap_importance"],
        "raw_ext_opp": opp["raw_external_opportunity"],
        "max_norm": opp["max_external_normalizer"],
        "O(s)": comp["O_external_opportunity"],
        "P(s)": r["priority_score"],
        "tier": r["priority_tier"],
    })

print("Worked Mathematical Verification (alpha=0.70, beta=0.30, gamma_eff=0.0, delta_eff=0.0):")
display(pd.DataFrame(worked_rows))

# Weight Sensitivity Comparison across all 4 presets
ws = partial_res["weight_sensitivity"]
print("\nWeight Sensitivity Top-5 Presets & Jaccard vs Baseline:")
ws_rows = []
for preset_name, top5 in ws["top_k_by_preset"].items():
    ws_rows.append({
        "preset": preset_name,
        "weights": str((BASELINE_WEIGHT_PRESETS[preset_name].alpha, BASELINE_WEIGHT_PRESETS[preset_name].beta)),
        "top5_jaccard_vs_baseline": ws["top_k_jaccard_vs_baseline"][preset_name],
        "top_5_skills": ", ".join(top5),
    })
display(pd.DataFrame(ws_rows))

Worked Mathematical Verification (alpha=0.70, beta=0.30, gamma_eff=0.0, delta_eff=0.0):


,rank,skill,req_type,gap_factor,req_share,pref_share,G(s),raw_ext_opp,max_norm,O(s),P(s),tier
0,1,English Language,required,1.0,0.9375,0.0,0.9375,48.9921,48.9921,1.0000,0.9563,Critical / High
1,2,Data Analysis,required,1.0,1.0000,0.0,1.0000,7.8907,48.9921,0.1611,0.7483,Critical / High
2,3,Tableau,required,1.0,0.6250,0.0,0.6250,0.9118,48.9921,0.0186,0.4431,Medium
3,4,Statistics,required,1.0,0.5625,0.0,0.5625,5.2964,48.9921,0.1081,0.4262,Medium
4,5,Data Visualization,required,1.0,0.5625,0.0,0.5625,2.2826,48.9921,0.0466,0.4077,Medium
5,6,SQL,required,0.5,1.0000,0.0,0.5000,4.7960,48.9921,0.0979,0.3794,Medium
6,7,Attention to Detail,required,1.0,0.3750,0.0,0.3750,11.0893,48.9921,0.2263,0.3304,Medium
7,8,Power BI,required,1.0,0.4375,0.0,0.4375,2.8361,48.9921,0.0579,0.3236,Medium



Weight Sensitivity Top-5 Presets & Jaccard vs Baseline:


,preset,weights,top5_jaccard_vs_baseline,top_5_skills
0,target_only,"(1.0, 0.0)",1.0000,"Data Analysis, English Language, Tableau, Stat..."
1,baseline_balanced_heuristic,"(0.7, 0.3)",1.0000,"English Language, Data Analysis, Tableau, Stat..."
2,equal_gap_opportunity,"(0.5, 0.5)",1.0000,"English Language, Data Analysis, Statistics, T..."
3,career_mobility_first,"(0.3, 0.7)",0.4286,"English Language, Data Analysis, Attention to ..."


## 3. Modules 4 & 6 — Cross-Occupation Portability & Honest Learning Path Ordering

Analyzes multi-occupation skill opportunity across the 53 occupations and verifies that learning-path sequencing preserves missing prerequisite/course boundaries rather than converting associative `CO_OCCURS_WITH` links into causal prerequisites.

In [6]:
key_skills = [
    ("SG-SK-0001", "Python"),
    ("SG-SK-0006", "SQL"),
    ("SG-SK-0024", "Statistics"),
    ("SG-SK-0022", "Power BI"),
    ("SG-SK-0027", "Machine Learning"),
]
opp_rows = []
for sid, name in key_skills:
    opp = multi_occ.analyze_skill_opportunities(sid)
    bridges = multi_occ.get_co_occurring_skills(sid, top_k=3)
    opp_rows.append({
        "skill_id": sid,
        "canonical_skill": name,
        "required_occupations": opp["required_occupation_count"],
        "preferred_only_occupations": opp["preferred_only_occupation_count"],
        "total_occupations": opp["total_occupation_count"],
        "raw_external_opportunity": opp["raw_external_opportunity"],
        "normalized_O(s)": opp["normalized_opportunity_score"],
        "top_co_occurring_bridges": ", ".join(b["canonical_skill"] for b in bridges),
    })
display(pd.DataFrame(opp_rows))

lp = partial_res["learning_path"]
print("\nLearning Path Strategy:", lp["ordering_strategy"])
print("Prerequisite Graph Status:", lp["prerequisite_graph_status"], "| Available:", lp["prerequisite_graph_available"])
print("Learning Resources Status:", lp["learning_resources_status"], "| Available:", lp["learning_resources_available"])
print("Limitation Notice:", lp["limitation_notice"])
for st in lp["stages"]:
    sample_names = [x["canonical_skill"] for x in st["items"][:4]]
    print(f"  - {st['stage']}: {st['count']} skills (sample: {sample_names})")

,skill_id,canonical_skill,required_occupations,preferred_only_occupations,total_occupations,raw_external_opportunity,normalized_O(s),top_co_occurring_bridges
0,SG-SK-0001,Python,5,8,13,5.7866,0.1159,"English Language, Statistics, Data Analysis"
1,SG-SK-0006,SQL,6,4,10,5.7960,0.1161,"English Language, Data Analysis, Python"
2,SG-SK-0024,Statistics,8,1,9,5.8589,0.1173,"English Language, Data Analysis, Report Writing"
3,SG-SK-0022,Power BI,3,4,7,3.2736,0.0656,"English Language, Data Analysis, Microsoft Excel"
4,SG-SK-0027,Machine Learning,2,0,2,2.0000,0.0401,"Python, English Language, Apache Spark"



Learning Path Strategy: transparent_priority_and_category_heuristic
Prerequisite Graph Status: UNAVAILABLE_IN_LOCAL_TAXONOMY | Available: False
Learning Resources Status: UNAVAILABLE_NO_VERIFIED_COURSE_CATALOG | Available: False
Limitation Notice: No real university course syllabus catalog, training provider, or empirical learning duration data is populated yet; therefore SkillGraph does not invent course names or hour estimates.
  - Stage 1: Core Target-Required Foundation: 2 skills (sample: ['English Language', 'Data Analysis'])
  - Stage 2: High-Portability & Category/Statistical Bridges: 20 skills (sample: ['Tableau', 'Statistics', 'Data Visualization', 'SQL'])
  - Stage 3: Role-Specific Preferred Differentiators: 2 skills (sample: ['Google Analytics', 'R (programming language)'])


## 4. Modules 7 & 8 — Relational Database Layer & REST API Smoke Verification

Verifies PostgreSQL host status, 17-table relational schema initialization in the offline SQLite execution mirror (`PRAGMA foreign_keys = ON`), round-trip persistence, and ASGI REST endpoint contracts (200/400/404/422).

In [7]:
pg_env = inspect_postgresql_environment()
db = RelationalDatabaseManager(builder, db_path=":memory:")
db.save_analysis_result(partial_res, engine, profiles["partial_da"])
fetched_analysis = db.fetch_analysis(partial_res["analysis_id"])
table_counts = db.list_tables_and_counts()

print("PostgreSQL host environment audit:", pg_env)
print(f"Relational tables initialized: {len(table_counts)} (DDL statements: {len(POSTGRESQL_DDL_STATEMENTS)})")
print("Round-trip analysis fetch ID match:", fetched_analysis["analysis_id"] == partial_res["analysis_id"])
display(pd.DataFrame([{"table": k, "rows": v} for k, v in table_counts.items()]))

client = SkillGraphTestClient(app)
r_health = client.get("/health")
r_occ = client.get("/occupations?search=Data")
r_occ_skills = client.get("/occupations/SG-OC-002/skills")
r_post = client.post("/skill-gap/analyze", json={
    "student_id": "SYNTHETIC_NB_API_01",
    "target_occupation": "Data Analyst",
    "skills": [
        {"skill": "Python", "proficiency_level": "intermediate"},
        {"skill": "SQL", "proficiency_level": "beginner"},
    ],
    "top_k": 5,
})
aid = r_post.json()["analysis_id"]
r_get_gap = client.get(f"/skill-gap/{aid}")
r_get_rec = client.get("/recommendations/SYNTHETIC_NB_API_01")
r_get_exp = client.get("/recommendations/SYNTHETIC_NB_API_01/explanations")
r_openapi = client.get("/openapi.json")
r_404 = client.post("/skill-gap/analyze", json={"student_id": "SYN_ERR", "target_occupation": "Unknown Role"})
r_400 = client.post("/skill-gap/analyze", json={"student_id": "SYN_ERR", "target_occupation": "Data Analyst", "skills": ["UnknownSkill99"], "strict_validation": True})
r_422 = client.post("/skill-gap/analyze", json={"student_id": "", "target_occupation": "Data Analyst"})

api_smoke_summary = {
    "health_status": r_health.status_code,
    "occupations_search_data_count": len(r_occ.json()),
    "da_required_count": len(r_occ_skills.json()["required_skills"]),
    "da_preferred_count": len(r_occ_skills.json()["preferred_only_skills"]),
    "post_analyze_status": r_post.status_code,
    "post_gap_score": r_post.json()["coverage_metrics"]["skill_gap_score"],
    "get_gap_status": r_get_gap.status_code,
    "get_rec_status": r_get_rec.status_code,
    "get_exp_status": r_get_exp.status_code,
    "openapi_paths_count": len(r_openapi.json()["paths"]),
    "openapi_schemas_count": len(r_openapi.json()["components"]["schemas"]),
    "negative_404_status": r_404.status_code,
    "negative_400_status": r_400.status_code,
    "negative_422_status": r_422.status_code,
}
print("\nAPI Smoke Verification:", json.dumps(api_smoke_summary, indent=2))

PostgreSQL host environment audit: {'psql_exists': True, 'pg_isready_exists': True, 'port_5432_status': '127.0.0.1:5432 - accepting connections', 'python_pg_drivers': {'psycopg2': False, 'psycopg': False, 'asyncpg': False, 'sqlalchemy': False}, 'env_vars_configured': {'SKILLGRAPH_DB_HOST': False, 'SKILLGRAPH_DB_PORT': False, 'SKILLGRAPH_DB_NAME': False, 'SKILLGRAPH_DB_USER': False, 'SKILLGRAPH_DB_PASSWORD': False}, 'live_pg_execution_ready': False, 'offline_sql_fallback': 'sqlite3_with_foreign_keys_enabled'}
Relational tables initialized: 17 (DDL statements: 23)
Round-trip analysis fetch ID match: True


,table,rows
0,skill_categories,18
1,skills,230
2,skill_aliases,325
3,industries,15
4,occupations,53
5,countries,20
6,job_postings,2000
7,job_skill_mentions,21588
8,occupation_skill_demand,1645
9,skill_co_occurrences,3987



API Smoke Verification: {
  "health_status": 200,
  "occupations_search_data_count": 3,
  "da_required_count": 21,
  "da_preferred_count": 5,
  "post_analyze_status": 200,
  "post_gap_score": 0.8863,
  "get_gap_status": 200,
  "get_rec_status": 200,
  "get_exp_status": 200,
  "openapi_paths_count": 8,
  "openapi_schemas_count": 13,
  "negative_404_status": 404,
  "negative_400_status": 400,
  "negative_422_status": 422
}


## 5. Export Measured Week 3 Benchmark Summary

Appends `week3_extensions` cleanly to `outputs/reports/skill_extraction_benchmark_report.json` while leaving all Week 1 & Week 2 JSON blocks untouched.

In [8]:
with open(BENCHMARK_REPORT_PATH, "r", encoding="utf-8-sig") as f:
    report_payload = json.load(f)

report_payload["week3_extensions"] = {
    "notebook_source": "research/notebooks/07_recommendation_evaluation.ipynb",
    "negation_module_benchmarks": {
        "spacy_8_case_assertions": f"{spacy_neg_eval['passed_assertions']}/{spacy_neg_eval['total_assertions']}",
        "spacy_8_case_accuracy": spacy_neg_eval["accuracy"],
        "dictionary_8_case_assertions": f"{dict_neg_eval['passed_assertions']}/{dict_neg_eval['total_assertions']}",
        "dictionary_8_case_accuracy": dict_neg_eval["accuracy"],
        "non_taxonomy_prohibited_clause_captured": prohib_clause["target_phrase"],
        "parser_unavailable_fallback_method": fallback_analysis["analysis_method"],
    },
    "technical_22_doc_benchmark_after_negation": tech_22_results,
    "corpus_2000_doc_modality_summary": {
        "spacy_strict_eval": {
            "tp": corpus_exact_eval["total_tp"],
            "fp": corpus_exact_eval["total_fp"],
            "fn": corpus_exact_eval["total_fn"],
            "precision": round(corpus_exact_eval["micro_precision"], 6),
            "recall": round(corpus_exact_eval["micro_recall"], 6),
            "f1": round(corpus_exact_eval["micro_f1"], 6),
            "elapsed_seconds": spacy_elapsed,
        },
        "modality_distribution_include_negated": corpus_modality_counts,
    },
    "skill_gap_and_priority_experiments": summary_rows,
    "worked_example_calculation_data_analyst": worked_rows[:6],
    "weight_sensitivity_evaluation": {
        "baseline_preset": ws["baseline_preset"],
        "top_k": ws["top_k"],
        "top_k_jaccard_vs_baseline": ws["top_k_jaccard_vs_baseline"],
        "top_k_by_preset": ws["top_k_by_preset"],
    },
    "multi_occupation_opportunity_stats": opp_rows,
    "learning_path_status": {
        "ordering_strategy": lp["ordering_strategy"],
        "prerequisite_graph_available": lp["prerequisite_graph_available"],
        "prerequisite_graph_status": lp["prerequisite_graph_status"],
        "learning_resources_available": lp["learning_resources_available"],
        "learning_resources_status": lp["learning_resources_status"],
        "stage_counts": {st["stage"]: st["count"] for st in lp["stages"]},
    },
    "relational_database_mirror": {
        "ddl_statement_count": len(POSTGRESQL_DDL_STATEMENTS),
        "table_count": len(table_counts),
        "table_row_counts": table_counts,
        "postgresql_host_environment": pg_env,
    },
    "api_smoke_checks": api_smoke_summary,
}

with open(BENCHMARK_REPORT_PATH, "w", encoding="utf-8", newline="\n") as f:
    json.dump(report_payload, f, indent=2, ensure_ascii=False)
    f.write("\n")

print("Updated benchmark report keys:", list(report_payload.keys()))

Updated benchmark report keys: ['experiment_id', 'milestone', 'ground_truth_provenance', 'week1_preserved_baselines', 'week2_2000_doc_evaluations', 'week2_22_doc_technical_benchmark', 'week2_9_category_error_analysis_2000_docs', 'scalability_benchmarks', 'week3_extensions']
